# A.8 同一條線索，放在哪裡會有差嗎？


把下面八筆活動室紀錄交給助理，問「攝影社的活動室代碼是什麼？只回答代碼」。紙上答案是 `M17`：代碼是房間的標籤，不必猜它代表哪一層樓。現在保留同一份材料，把提供答案的那筆紀錄移到資料的開頭、中間或結尾，三次都應回答 `M17`。這樣可以問一件比「輸入放得下」更進一步的事：位置改了，助理還能穩定用到那條線索嗎？

| 紀錄標籤 | 原文 |
| --- | --- |
| T | 攝影社的活動室代碼是 M17。 |
| U1 | 棋藝社的活動室代碼是 B04。 |
| U2 | 合唱社的活動室代碼是 C26。 |
| U3 | M17 在二樓。 |
| U4 | B04 在一樓。 |
| U5 | C26 在三樓。 |
| U6 | 美術社的活動室代碼是 E31。 |
| U7 | E31 在二樓。 |

T、U1 這些標籤讓我們能追蹤紀錄，並不是活動室代碼。這一題只需要 T；其他紀錄雖與活動室有關，都不能替代攝影社與 `M17` 的對應。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-A-clue-position.svg")))

圖中每個方框是一筆紀錄，不是一個 token。綠框就是紀錄 T，框裡寫的是它提供的代碼 `M17`；灰框是 U1 到 U7。先將 T 放在第一筆；再拿起同一張 T，插到第四筆；最後移到第八筆。U1 到 U7 的相對順序保持不變，沒有新增公告、改寫線索或另換問題。這裡的「首、中、尾」指**資料段**的位置，問題始終放在資料段之後。因此尾端版本的線索比較靠近問題，這正是要觀察的位置差異，不是把問題也一起搬動。下面的程式照這個規則排出三版，並核對它們真的只差在 T 的位置：


In [ ]:
from tiny_perceptron.data import ByteTokenizer

clue = "攝影社的活動室代碼是 M17。"
others = [
    "棋藝社的活動室代碼是 B04。",
    "合唱社的活動室代碼是 C26。",
    "M17 在二樓。",
    "B04 在一樓。",
    "C26 在三樓。",
    "美術社的活動室代碼是 E31。",
    "E31 在二樓。",
]
question = "攝影社的活動室代碼是什麼？只回答代碼。"
tok = ByteTokenizer()
for name, index in [("首", 0), ("中", 3), ("尾", 7)]:
    records = others[:index] + [clue] + others[index:]
    prompt = "\n".join(records) + "\n" + question
    same_order = [r for r in records if r != clue] == others
    print(name, "T 在第", records.index(clue) + 1, "筆 token 數", len(tok.encode(prompt)), "其餘順序不變", same_order)

`others[:index] + [clue] + others[index:]` 把 U1 到 U7 從第 index 項切成兩段（切片見 [W.2](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.2)），把 T 插在中間：index＝0 放最前面，3 放在 U3 之後，7 放在最後。`"\n".join(records)` 用換行把八筆接成資料段，再接上問題。`records.index(clue)` 找出 T 在清單裡的位置，從 0 起算，所以加 1 才是第幾筆。`same_order` 先用生成式拿掉 T，再和原本的 `others` 比，三版都是 True。三版的 token 數都是 277：本課的 `ByteTokenizer`（[6.6](https://birdhackor.github.io/tiny-perceptron-vlm/6.6.html)）每個 byte 一個 token，三版內容相同、只換順序，token 數必然相同。換成會合併常見片段的 tokenizer（第 6 章）時，紀錄接縫處可能切得不同，仍要逐版實際數。

若真的拿模型比較，三版還要使用同一句問題、同一句線索，以及相同的新回答上限。依 [A.7](https://birdhackor.github.io/tiny-perceptron-vlm/A.7.html) 的方法，先套上對話格式組成完整輸入，再核對三版的 token 數一致，且八筆紀錄都完整留下。模型的參數、選字規則（每一步怎麼從候選中挑出 token）與其他設定也保持固定。若重排後 token 數不同，先調整三版共用的紀錄分隔格式，重新核對長度再比較，不能把多出的長度當成位置效應。

紙上可以先畫三欄「首、中、尾」，每欄填上期待答案 `M17`。之後若有實際生成，再各自記下原始回答、代碼是否正確，以及回答是否完整結束。例如 `M17` 是正確內容；`B04` 是取錯紀錄；只輸出 `M` 就用完回答上限，則還有輸出未完成的問題。把這些差別留下來，比只記「成功／失敗」更容易找出要檢查的地方。

還可以另外準備只含 T 與同一問題的短版。短版若也答錯，就先檢查模型是否懂這個提問及代碼格式；短版答對而長版失敗，才有理由繼續查長度、位置或干擾。短版是另一個診斷條件，不與三個等長版本混算。

在多題比較中，若中間位置的答對率常比首尾低，可以描述「這個模型在這組題目上對位置敏感」；以線索位置為橫軸、答對率為縱軸畫出來，兩端高、中間低，形狀像字母 U，所以常叫 U 形曲線。也可能三處都答對，或出現別的排列。位置測試的用途是把差異量出來，沒有預先指定每個模型都必須出現 U 形。即使三處都成功，也只驗收了這種單筆取回；下一節會問，需要多筆資訊時又該怎麼查。

練習把程式的 `("中", 3)` 改成 `("中", 5)`，先預測：T 變成第 6 筆，token 數仍是 277，其餘順序仍不變；再執行核對。

<details>
<summary>補充：來源與例子的界線</summary>

本節「只移動線索位置、其他都不變」的比較方式參考 Liu et al., [*Lost in the Middle: How Language Models Use Long Contexts*，arXiv:2307.03172v3](https://arxiv.org/pdf/2307.03172v3)，§2.1–2.3、§3.1。原論文在多個當時模型與設定觀察到首尾較好、中間較差，也有部分合成取回任務接近全對。本節的八筆紀錄是教學用紙上材料，沒有重現原論文評測，也沒有提供本課模型的位置成績。

</details>
